# V2a-RSNs Linear BSS Decomposition

This notebook decomposes the V2a-RSN calcium time series with FastICA, Infomax, SOBI, and JADE. It saves only decomposition artifacts needed by `clustering_methods.ipynb`: IC time courses, spectra, mixing matrix, mean vector, and metadata.

Artifacts are saved under `outputs/linear/<dataset_group>/<data_name>/<method>/` when `SAVE_DECOMPOSITION_OUTPUTS = True`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    p
    for p in candidates
    if (p / "pyproject.toml").exists() and (p / "src" / "bss_notebook.py").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
CORE_DIR = SRC_DIR / "core"
for path in (SRC_DIR, CORE_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from bss_notebook import (
    BSS_METHODS,
    available_datasets,
    load_traces,
    output_directory,
    run_bss_decomposition,
    summarize_decomposition_results,
)

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PROJECT_ROOT


In [ ]:
available_datasets(PROJECT_ROOT)


In [ ]:
# Dataset and method selection
DATASET_KEY = "v2a-RSNs/220127_F4_run2_fluorescence"

# Use list(BSS_METHODS) to run all four methods, or choose a subset such as ["fastica"].
METHODS_TO_RUN = ["fastica", "infomax"] # list(BSS_METHODS)

# None uses one component per neuron/time series for this decomposition stage.
N_COMPONENTS = None
DECOMPOSITION_TOL = 0.0001
DECOMPOSITION_MAX_ITER = 500
DECOMPOSITION_RANDOM_STATE = 0

SAVE_DECOMPOSITION_OUTPUTS = True


In [ ]:
dataset, traces = load_traces(DATASET_KEY, PROJECT_ROOT)
n_neurons, n_frames = traces.shape

# Use number of neurons/time series by default so clustering sees one IC slot per trace.
if N_COMPONENTS is not None:
    n_components = N_COMPONENTS
else:
    n_components = n_neurons

n_components = min(n_components, n_neurons, n_frames)

print(f"dataset: {dataset.key}")
print(f"trace file: {dataset.trace_path.relative_to(PROJECT_ROOT)}")
print(f"traces: neurons={n_neurons}, frames={n_frames}")
print(f"methods: {METHODS_TO_RUN}")
print(f"n_components: {n_components}")
for method in METHODS_TO_RUN:
    out_dir = output_directory(method, dataset.data_name, PROJECT_ROOT, dataset_group=dataset.group)
    print(f"{method} decomposition dir: {out_dir.relative_to(PROJECT_ROOT)}")


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
time_slice = slice(0, min(300, n_frames))
offset = np.nanstd(traces[:, time_slice]) * .4
if not np.isfinite(offset) or offset == 0:
    offset = 1.0
for neuron_idx in range(min(6, n_neurons)):
    ax.plot(
        np.arange(time_slice.start, time_slice.stop),
        traces[neuron_idx, time_slice] + neuron_idx * offset,
        lw=0.8,
        label=f"neuron {neuron_idx}",
    )
ax.set_title("Raw traces")
ax.set_xlabel("frame")
ax.legend(loc="upper right", ncols=3);


In [ ]:
print(f"
{'='*60}")
print(f"Running {len(METHODS_TO_RUN)} decompositions with {n_components} components...")
print(f"{'='*60}
")

results = {}
for idx, method in enumerate(METHODS_TO_RUN, 1):
    print(f"[{idx}/{len(METHODS_TO_RUN)}] Fitting {method.upper()}...", flush=True)
    results[method] = run_bss_decomposition(
        dataset_key=DATASET_KEY,
        method=method,
        n_components=n_components,
        save_outputs=SAVE_DECOMPOSITION_OUTPUTS,
        project_root=PROJECT_ROOT,
        tol=DECOMPOSITION_TOL,
        max_iter=DECOMPOSITION_MAX_ITER,
        random_state=DECOMPOSITION_RANDOM_STATE,
    )
    print(f"  {method.upper()} completed
", flush=True)

print(f"{'='*60}")
print("All decompositions completed successfully!")
print(f"{'='*60}
")

summarize_decomposition_results(results)


In [ ]:
component_count = min(8, n_components)
fig, axes = plt.subplots(len(results), 1, figsize=(14, 2.0 * len(results)), sharex=True)
if len(results) == 1:
    axes = [axes]
for ax, (method, result) in zip(axes, results.items()):
    for comp_idx in range(component_count):
        ax.plot(result.ic_comps[:, comp_idx] + comp_idx * 3, lw=0.7, label=f"IC {comp_idx}")
    ax.set_title(f"{method.upper()} first {component_count} components")
axes[-1].set_xlabel("frame");


In [ ]:
for method, result in results.items():
    if result.saved_paths:
        print(f"{method}:")
        for label, path in result.saved_paths.items():
            print(f"  {label}: {path.relative_to(PROJECT_ROOT)}")
    else:
        print(
            f"{method}: not saved; set SAVE_DECOMPOSITION_OUTPUTS = True and rerun the decomposition cell "
            f"to write artifacts under {result.output_dir.relative_to(PROJECT_ROOT)}"
        )
